# Error Analysis on LIAR / LIAR 上的错误分析

`07`、`08` 回答了"分数是多少"：只看文字的模型 AUC 约 0.68，DistilBERT + TF-IDF + Haiku 的组合（5c）达到 0.73。这里回答 **"错在哪里、为什么错"**：
1. 哪些真假等级、哪类陈述最难？
2. DistilBERT（语言模式）和 Haiku（世界知识）各自判对了什么？组合是怎么利用两者的？
3. 最自信的错误是什么样的？
4. 模型是否对某个党派、某个话题有系统性的偏差？

**这里只描述已经固定的模型，不调整任何东西**，所以在测试集上分析不存在"调参"的问题。说话人、党派、话题等信息 **只用于分组**，从未作为模型的输入。

`07` and `08` measured how well the models do; this notebook asks where and why they fail. It only describes fixed models on LIAR test; speaker, party and subject are used for grouping only, never as inputs.

## 1. Setup / 环境

- **组合模型**：5c 选中的 DistilBERT + TF-IDF + Haiku，用 DistilBERT 种子 42 的版本（3 个种子在 91% 的陈述上判断完全一致，见下面）。
- **单个模型**：为了比较"谁判对了"时公平，每个单独的模型都先在 val 上校准（一个特征的逻辑回归，和 5c 的单模型候选完全相同），再用 0.5 作为分界线。否则 Haiku 的原始概率偏向"假"，会显得比实际更差。

The combined model is the 5c choice (DistilBERT seed 42 version). Single models are calibrated on val first so that "who is right" uses fair thresholds.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src import error_analysis as E
from src.external import load_liar, load_liar_metadata

LLM = Path('../results/llm')
ORDER = ['pants-fire', 'false', 'barely-true', 'half-true', 'mostly-true', 'true']
pd.set_option('display.width', 220, 'display.max_colwidth', 160)

liar = load_liar().set_index('doc_id')
meta = load_liar_metadata().set_index('doc_id')
rec = json.loads((LLM / 'combination_test.json').read_text())
ids = rec['doc_ids']
y = liar.loc[ids, 'label']
label6 = liar.loc[ids, 'label6']
text = liar.loc[ids, 'text']

combined = {s: pd.Series(rec['results'][f'seed {s}']['prob'], index=ids) for s in (42, 43, 44)}
P = {'combined': combined[42],
     'DistilBERT': E.calibrated('distilbert_seed42').loc[ids],
     'Haiku': E.calibrated('haiku').loc[ids],
     'TF-IDF': E.calibrated('tfidf').loc[ids]}

same = (combined[42] > 0.5).eq(combined[43] > 0.5) & (combined[42] > 0.5).eq(combined[44] > 0.5)
print(f'The three combined seeds give the same verdict on {100 * same.mean():.1f}% of statements')
pd.DataFrame({name: {'accuracy %': round(100 * ((p > 0.5) == y).mean(), 1),
                     'predicted real %': round(100 * (p > 0.5).mean(), 1)} for name, p in P.items()}).T

The three combined seeds give the same verdict on 91.3% of statements


,accuracy %,predicted real %
combined,67.0,57.5
DistilBERT,65.5,56.5
Haiku,63.1,56.0
TF-IDF,62.9,57.4


校准之后，三个单独的模型判为真的比例都在 56–57% 左右，接近真实的 56.7%，准确率 63–66%；组合是 67%。

After calibration all single models predict ~56–57% real (true share 56.7%); the combination reaches 67%.

## 2. Which truth levels are hard? / 哪些真假等级最难

二分类的分界线在 barely-true 和 half-true 之间。离分界线越近的等级，越难判断。

In [2]:
pd.DataFrame({'n': label6.value_counts().reindex(ORDER),
              'binary label': [0, 0, 0, 1, 1, 1],
              **{f'{name}: accuracy %': ((p > 0.5) == y).groupby(label6).mean().reindex(ORDER).mul(100).round(1)
                 for name, p in P.items()}})

,n,binary label,combined: accuracy %,DistilBERT: accuracy %,Haiku: accuracy %,TF-IDF: accuracy %
label6,,,,,,
pants-fire,92,0,76.1,70.7,76.1,64.1
false,250,0,63.6,61.6,57.2,54.8
barely-true,214,0,51.4,54.7,51.9,54.7
half-true,267,1,59.2,60.7,58.1,62.9
mostly-true,249,1,78.7,74.3,72.7,73.1
true,211,1,79.1,74.9,71.1,68.2


- **两端最容易**：pants-fire（彻底的谎言）76.1%，mostly-true 78.7%，true 79.1%（组合模型）。
- **分界线两侧最难**：barely-true（"几乎不属实"，算假）51.4%，half-true（"一半属实"，算真）59.2%。这两级本身就很接近，标签的差别往往取决于 PolitiFact 编辑的判断，而不是文字里可以看出的东西。
- **组合的提升集中在两端**：pants-fire、false、mostly-true、true 这 4 个等级上，组合都不低于任何单个模型。但在分界线两侧的 barely-true 和 half-true 上，组合反而 **略低于** DistilBERT 和 TF-IDF（例如 half-true：59.2% 对 60.7% 和 62.9%）。组合让模型在"明显的真"和"明显的假"上更有把握，但对靠近分界线的陈述帮不上忙。

The extremes are easiest (76–79% for the combination); barely-true (51.4%) and half-true (59.2%), on either side of the binary boundary, are hardest. The combination gains on the four outer levels but is slightly below DistilBERT and TF-IDF on the two boundary levels.

## 3. Which statements are hard? / 哪类陈述最难

用几条简单的规则给陈述分类（`src/error_analysis.py`，都是粗略的规则，只用于描述）：
- **含数字或数量**（"3 percent"、"doubled"、"half"）；
- **绝对化的说法**（"never"、"every"、"first"、"record"）；
- **依赖上下文的指代**（"he"、"his"、"this bill"、"my opponent"）：不知道指的是谁 / 什么；
- **以 "Says" 开头**：PolitiFact 对广告、社交媒体帖子的转述。

AUC 只比较组内的排序，不受各组真陈述比例不同的影响，所以下面主要看 AUC。

In [3]:
F = E.statement_features(text)
auc_cols = [f'{n}: AUC' for n in ('combined', 'DistilBERT', 'Haiku')]
tables = {c: E.group_table(y, {k: P[k] for k in ('combined', 'DistilBERT', 'Haiku')}, F[c]) for c in F.columns}
pd.concat({c: t[['n', 'real %'] + auc_cols] for c, t in tables.items()})

n  real %  combined: AUC  DistilBERT: AUC  Haiku: AUC
length                      1-12 words   339.0    52.5          0.683            0.645       0.649
                            13-20        534.0    56.2          0.757            0.711       0.658
                            21-30        325.0    59.1          0.724            0.671       0.660
                            31+           85.0    67.1          0.714            0.649       0.632
has number / quantity       False        726.0    50.6          0.713            0.657       0.655
                            True         557.0    64.6          0.714            0.665       0.644
absolute claim              False        983.0    53.9          0.709            0.670       0.647
                            True         300.0    65.7          0.781            0.709       0.688
context-dependent reference False       1009.0    55.1          0.748            0.695       0.677
                            True         274.0    62.4          0.653            0.635       0.568
starts with "Says"          False        991.0    57.8          0.727            0.679       0.658
                            True         292.0    52.7          0.729            0.694       0.658

**最明显的规律：依赖上下文的陈述最难，而且主要难在 Haiku 身上。**

| | 组合 AUC | DistilBERT AUC | Haiku AUC |
|---|---:|---:|---:|
| 没有"他 / 这项法案"之类的指代 | 0.748 | 0.695 | 0.677 |
| **有这类指代**（21% 的陈述） | **0.653** | 0.635 | **0.568** |

Haiku 靠世界知识判断，但"他任内失业率翻了一倍"里的"他"是谁、"任内"是哪几年都不知道，知识就用不上，AUC 降到接近随机的 0.568。这直接证实了前面分析的瓶颈：**很多陈述缺的不是证据，而是背景**。

其他发现：
- **短陈述更难**（1–12 个词：组合 AUC 0.683；13–20 个词：0.757）。信息越少越难。
- **绝对化的说法反而更容易**（0.781）：一个反例就能推翻，模型的知识足够判断。
- **含数字的陈述并不比其他陈述更难**（0.713 对 0.714），和我之前的猜测不同。但下面第 5 节会看到，数字在 *最自信的错误* 里非常集中。

Context-dependent statements are the hardest, mainly for Haiku (AUC 0.568 vs 0.677): world knowledge is useless when the claim does not say who or what it is about. Short statements are harder; absolute claims are easier; numbers do not lower the AUC overall.

## 4. A dataset artefact: "On …" fragments / 数据集的一个问题：以 "On …" 开头的片段

有些"陈述"根本不是一句话，例如 *On toll roads.*、*On the Bush tax cuts.*。这些来自 PolitiFact 的 **Flip-O-Meter**：评的是政治人物 **有没有改变立场**（Full Flop / Half Flip / No Flip），不是一个说法是否属实，LIAR 把它们混进了六级标签里。只看这段文字，任何模型都不可能判断。

In [4]:
frag = text.str.match(r'^\s*On\s') & (text.str.split().str.len() <= 15)
print(f'{frag.sum()} fragments ({100 * frag.mean():.1f}% of test) | labels:', label6[frag].value_counts().to_dict())
print('Combined accuracy on fragments:', round(100 * ((P["combined"][frag] > 0.5) == y[frag]).mean(), 1), '%\n')
print(text[frag].head(8).to_string())

31 fragments (2.4% of test) | labels: {'false': 15, 'half-true': 13, 'true': 3}
Combined accuracy on fragments: 51.6 %

doc_id
liar_8047                        On residency requirements for public workers
liar_816                   On which team hes rooting for in the World Series.
liar_2885                                                On banning earmarks.
liar_10147    On an income cap for recipients of the popular HOPE scholarship
liar_2277                             On running a civil and polite campaign.
liar_715                               On whether global warming is man-made.
liar_4016                          On pay for fired Milwaukee police officers
liar_603                                                On the Bush tax cuts.


31 条（2.4%），组合模型在它们上面的准确率 51.6%，等于瞎猜。这部分是 LIAR 本身的噪声，任何方法都无法改进；它也拉低了所有模型的上限。

31 statements (2.4%) are Flip-O-Meter items about changing positions, not truth claims; accuracy on them is at chance. This is label noise in LIAR itself.

## 5. Who knows what: DistilBERT vs Haiku / 谁知道什么

两个模型（都已校准）在每条陈述上谁判对了。

In [5]:
q = E.agreement_quadrants(y, P['DistilBERT'], P['Haiku'], 'DistilBERT', 'Haiku')
quad = pd.crosstab(q, label6)[ORDER]
quad.insert(0, 'n', q.value_counts())
quad['combined accuracy %'] = ((P['combined'] > 0.5) == y).groupby(q).mean().mul(100).round(1)
quad.loc[['both right', 'only DistilBERT right', 'only Haiku right', 'both wrong']]

label6,n,pants-fire,false,barely-true,half-true,mostly-true,true,combined accuracy %
row_0,,,,,,,,
both right,569,52,94,61,103,144,115,98.8
only DistilBERT right,272,13,60,56,59,41,43,57.0
only Haiku right,241,18,49,50,52,37,35,58.9
both wrong,201,9,47,47,53,27,18,0.5


- 两个模型 **在 40% 的陈述上意见不同**（只有一个对：272 + 241 = 513 条），两边各对一半左右。这就是组合有用的原因：两者犯的不是同一种错误。
- 在意见不同的陈述上，组合判对了约 57–59%；在两个都对的陈述上几乎全对，在两个都错的陈述上几乎全错（201 条，16%）。**组合的提升完全来自"意见不同"的那 40%**，它没有凭空创造新的信息。
- 两个都错的 201 条，集中在 false、barely-true、half-true 这些靠近分界线的等级。

The two models disagree on 40% of statements, each right about half the time; the combination gets ~58% of those right, almost all of the agreed-right ones, and almost none of the 16% both get wrong. Its gain comes entirely from the disagreements.

In [6]:
show = pd.DataFrame({'label6': label6, 'combined': P['combined'].round(2), 'DistilBERT': P['DistilBERT'].round(2),
                     'Haiku': P['Haiku'].round(2), 'statement': text.str[:140]})
rng = np.random.default_rng(0)
for k in ('only Haiku right', 'only DistilBERT right'):
    sub = show[q == k]
    print(f'--- {k}: 8 random examples ---')
    print(sub.loc[rng.choice(sub.index, 8, replace=False)].to_string(), '\n')

--- only Haiku right: 8 random examples ---
                 label6  combined  DistilBERT  Haiku                                                                                                                                     statement
liar_3029   mostly-true      0.59        0.49   0.69                                                    The state is able to sharply curtail contract bargaining rights for state employee unions.
liar_4719     half-true      0.66        0.48   0.70                                                              Proposition 2 on Novembers Texas ballot does not cost state taxpayers any money.
liar_13270  barely-true      0.43        0.50   0.44  We know that more than half of Hillary Clintons meetings while she was secretary of state were given to major contributors to the Clinton Fo
liar_3414         false      0.46        0.55   0.44                                                                                 We now work the longest hours of any people

这些随机例子只用于直观感受，8 条看不出非常清楚的分工，不能据此下结论。比较可靠的是 **定量** 的证据：
- Haiku 在 **依赖上下文** 的陈述上几乎失效（第 3 节，AUC 0.568），而 DistilBERT 下降得少（0.635）：不知道"他"是谁时，知识用不上，但说法的措辞仍然有信息。
- Haiku 在 **网络传言** 上最强、在 **政客的说法** 上较弱（第 7 节）。
- 还可以注意到，Haiku 的分数集中在少数几个值上：它最常回答 72、25、15、35、75，**6 个值就覆盖了约 90% 的陈述**（校准后对应 0.69、0.44、0.36、0.50、0.70）。所以它的排序比 DistilBERT 粗糙得多，很多陈述在 Haiku 看来是"同分"的。

The random examples are illustrative only; eight of them show no clean division of labour. The reliable evidence is quantitative: Haiku collapses on context-dependent statements (section 3) and is strongest on viral rumours (section 7). Its calibrated scores also take only a few distinct values, because it answers with a handful of round integers.

## 6. The most confident mistakes / 最自信的错误

In [7]:
print('=== Fake statements the combination was most sure were TRUE ===')
print(show[y == 0].sort_values('combined', ascending=False).head(10).to_string(), '\n')
print('=== True statements the combination was most sure were FAKE ===')
print(show[y == 1].sort_values('combined').head(10).to_string())

=== Fake statements the combination was most sure were TRUE ===
                 label6  combined  DistilBERT  Haiku                                                                                                                                     statement
liar_9231         false      0.93        0.81   0.70  More than 1,000 American soldiers have lost their lives in Afghanistan in the last 27 months. This is more than the combined total of the ni
liar_8151         false      0.92        0.71   0.69                            Of the roughly 15 percent of Americans who dont have health insurance, half of them made more than $50,000 a year.
liar_9256         false      0.90        0.72   0.77                                                                                    Some of the wealthiest Americans are African-American now.
liar_11080  barely-true      0.89        0.80   0.69                            Texas high school graduation rate went from 27th in the country in 2002, to 

In [8]:
fp = P['combined'][y == 0].sort_values(ascending=False).head(50).index
fn = P['combined'][y == 1].sort_values().head(50).index
pd.DataFrame({
    'among the 50 most confident errors': [F.loc[fp, 'has number / quantity'].mean(), (label6[fn] == 'half-true').mean()],
    'among all statements of that label': [F.loc[y[y == 0].index, 'has number / quantity'].mean(),
                                           (label6[y == 1] == 'half-true').mean()],
}, index=['fake judged true: share with numbers', 'true judged fake: share half-true']).mul(100).round(1)

,among the 50 most confident errors,among all statements of that label
fake judged true: share with numbers,72.0,35.4
true judged fake: share half-true,62.0,36.7


**两种自信的错误，各有一个清楚的模式：**

1. **把假的判成真：72% 含有具体数字**（所有假陈述中只有 35%）。例如"得克萨斯州的高中毕业率从全国第 27 位升到第 2 位"、"亚特兰大犯罪率下降了 14%，暴力犯罪下降了 22.7%"。**精确的数字听起来很权威**，两个模型都被说服了；但错误往往藏在细节里（排名的口径、时间段的选取），只有查到原始数据才能发现。
2. **把真的判成假：62% 是 half-true**（所有"真"陈述中只有 37%）。例如"奥巴马向墨西哥毒贩提供了枪支"（指"快速与狂暴"行动，评级 half-true）。这些说法 **听起来很夸张、像谣言**，但在技术上部分成立。模型判"假"其实并不离谱，问题出在二分类把 half-true 算作"真"。

1. Confident false positives are dominated by precise statistics (72% contain numbers vs 35% of all fakes): exact numbers sound authoritative, while the error hides in the details.
2. Confident false negatives are dominated by half-true claims (62% vs 37%): sensational-sounding but technically partly true; the binary mapping counts them as true.

## 7. Bias by party and subject / 党派和话题上的偏差

如果模型对某个党派系统性地偏宽松或偏严格，"判为真的比例"会明显偏离"实际真的比例"。

In [9]:
party = meta.loc[ids, 'party']
three = {k: P[k] for k in ('combined', 'DistilBERT', 'Haiku')}
E.bias_by_group(y, three, party)

,n,actual real %,combined: predicted - actual,DistilBERT: predicted - actual,Haiku: predicted - actual
democrat,410.0,67.3,3.4,-2.7,1.7
none,216.0,44.9,3.2,4.6,0.5
republican,580.0,54.1,-1.9,-0.5,-2.9


In [10]:
E.group_table(y, three, party)[['n', 'real %'] + auc_cols]

,n,real %,combined: AUC,DistilBERT: AUC,Haiku: AUC
democrat,410.0,67.3,0.700,0.669,0.617
none,216.0,44.9,0.783,0.695,0.728
republican,580.0,54.1,0.697,0.674,0.614


- **没有明显的党派偏差**：三个模型对民主党、共和党、无党派陈述的"判为真比例 − 实际比例"都在 ±5 个百分点以内。
- 但 **Haiku 在两党政治人物的陈述上明显更弱**（AUC 约 0.61–0.62），在无党派来源（`none`：多是连锁邮件、网络传言、社交媒体帖子）上最强（0.728）。网络传言往往是广为流传、已被反复辟谣的说法，正是大模型的世界知识最有用的地方；而政客的说法更多是具体的政策数字和口径之争。

No clear party bias (all within ±5 points). Haiku is much stronger on viral rumours and chain emails (party "none", AUC 0.728) than on politicians' claims (~0.61): widely circulated rumours are where world knowledge helps most.

In [11]:
subject = meta.loc[ids, 'subject'].str.split(',').str[0]          # 第一个话题标签 / first subject tag
E.group_table(y, three, subject, min_n=40)[['n', 'real %'] + auc_cols].sort_values('combined: AUC')

,n,real %,combined: AUC,DistilBERT: AUC,Haiku: AUC
candidates-biography,75.0,48.0,0.657,0.636,0.613
foreign-policy,40.0,52.5,0.659,0.679,0.551
economy,105.0,71.4,0.698,0.616,0.667
federal-budget,54.0,66.7,0.698,0.568,0.694
crime,43.0,60.5,0.719,0.713,0.621
health-care,86.0,51.2,0.752,0.716,0.634
education,74.0,62.2,0.759,0.760,0.670
elections,61.0,59.0,0.858,0.702,0.800


- **最难的话题是 candidates-biography**（候选人履历，组合 AUC 0.657）和 **foreign-policy**（外交，0.659）。候选人履历多是地方性的细节，模型没有相关知识；外交上 Haiku 尤其弱（0.551）。
- **最容易的是 elections**（选举，0.858）：选举舞弊、投票规则等说法反复出现，常常是广为流传的传言，Haiku 在这里也很强（0.800）。
- 每个话题只有 40–105 条陈述，这些数字的波动较大，只作为参考。

Candidate biographies (AUC 0.657) and foreign policy (0.659) are hardest; elections are easiest (0.858). Each subject has only 40–105 statements, so these are indicative only.

## 8. Conclusions / 结论

**1. 最大的瓶颈是"缺少背景"，不是"缺少证据"。**
含有"他""这项法案""我的对手"之类指代的陈述占 21%，组合模型在上面的 AUC 从 0.748 降到 0.653；Haiku 降得最厉害（0.677 → 0.568，接近随机）。不知道说的是谁、是什么时候，世界知识和检索到的证据都用不上。这解释了 5b 中维基百科证据为什么帮助不大。

**2. 一部分错误来自 LIAR 的标签本身，任何模型都无法修正。**
- **分界线两侧最难**：barely-true（51%）和 half-true（59%）接近瞎猜。最自信的"把真判成假"中 62% 是 half-true：听起来夸张、像谣言，但技术上部分成立。
- **2.4% 的"陈述"根本不是说法**：以 "On …" 开头的片段来自 Flip-O-Meter，评的是有没有改变立场，模型在它们上面的准确率 52%。

**3. 精确的数字最容易骗过模型。** 最自信的"把假判成真"中 72% 含有具体数字（所有假陈述中是 35%），例如州排名、犯罪率的百分比。数字听起来权威，而错误藏在口径和时间段里，只有查到原始数据才能发现。

**4. 组合的提升来自两个模型"意见不同"的那 40% 陈述。** DistilBERT 和 Haiku 各自判对一半左右，组合判对其中约 58%；两个都错的 16% 组合几乎从未救回。组合的提升集中在明显的真和明显的假上，对分界线附近的陈述帮不上忙。

**5. 没有发现明显的党派偏差**（判为真的比例和实际比例相差都在 ±5 个百分点以内）。Haiku 在网络传言上最强（AUC 0.728），在政客的说法上较弱（约 0.61），因为广为流传的谣言正是它的知识最能覆盖的地方。

**对项目的意义**：要在 LIAR 上进一步提高，需要的是 **陈述的背景**（谁说的、什么时候、针对什么）和 **原始数据**（统计数字的来源），而不是更大的模型或更多的维基百科段落。同时，相当一部分错误来自 half-true 的归类和 Flip-O-Meter 片段，这部分是数据集的上限。

**局限**：陈述类型是用简单的规则识别的，会有误判；话题和说话人的分组样本较小；例子是随机抽取的，只作为直观说明。

---

1. **Missing context, not missing evidence, is the main bottleneck**: on the 21% of statements with references like "he" or "this bill", the combined AUC falls from 0.748 to 0.653, and Haiku's to 0.568.
2. **Part of the error is label noise**: barely-true and half-true are near chance; 62% of the most confident false negatives are half-true; 2.4% of "statements" are Flip-O-Meter fragments rated on consistency, not truth.
3. **Precise numbers fool the models**: 72% of the most confident false positives contain specific figures (vs 35% of all fakes).
4. **The combination's gain comes from the 40% of statements where DistilBERT and Haiku disagree**; it rarely rescues the 16% both get wrong.
5. **No clear party bias** (within ±5 points); Haiku is strongest on viral rumours (AUC 0.728) and weaker on politicians' claims (~0.61).